In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
import shutil
from sklearn.model_selection import train_test_split
import cv2

In [ ]:
# Set paths based on your folder structure
base_path = '/content/drive/MyDrive/Colab Notebooks/Corrosion'

In [ ]:
# Clean up old test/train data before starting
import shutil

train_dir = '/content/train_data'
test_dir = '/content/test_data'

# Remove old directories if they exist
if os.path.exists(train_dir):
    shutil.rmtree(train_dir)
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)

print("Cleaned up old directories")

Cleaned up old directories


In [ ]:
# Create directories for organized data
train_dir = '/content/train_data'
test_dir = '/content/test_data'
os.makedirs(train_dir + '/rust', exist_ok=True)
os.makedirs(train_dir + '/no_rust', exist_ok=True)
os.makedirs(test_dir + '/rust', exist_ok=True)
os.makedirs(test_dir + '/no_rust', exist_ok=True)

In [ ]:
# Load and organize images with better filtering
rust_images = [f for f in os.listdir(base_path + '/rust')
               if f.lower().endswith(('.jpg', '.png', '.jpeg'))
               and not f.startswith('.')
               and not f.startswith('_')]

no_rust_images = [f for f in os.listdir(base_path + '/no_rust')
                  if f.lower().endswith(('.jpg', '.png', '.jpeg'))
                  and not f.startswith('.')
                  and not f.startswith('_')]

print(f"Total rust images: {len(rust_images)}")
print(f"Total no_rust images: {len(no_rust_images)}")

# Add verification
if len(rust_images) < 20:
    print(f"⚠️ WARNING: Only {len(rust_images)} rust images found. Need at least 20!")
if len(no_rust_images) < 20:
    print(f"⚠️ WARNING: Only {len(no_rust_images)} no_rust images found. Need at least 20!")

# Show sample filenames
print(f"\nSample rust images: {rust_images[:3]}")
print(f"Sample no_rust images: {no_rust_images[:3]}")

Total rust images: 24
Total no_rust images: 24

Sample rust images: ['002_cql5orz2.p0y.jpg', '004_liem2hdg.3w2.jpg', '004_i1dmciub.oop.jpg']
Sample no_rust images: ['003_zomak3wq.43c.jpg', '002_yoamnba2.wdq.jpg', '006_ri0wotdb.hsh.jpg']


In [ ]:
# Randomly select 10 images from each class for testing
np.random.seed(42)
test_rust = np.random.choice(rust_images, 10, replace=False)
test_no_rust = np.random.choice(no_rust_images, 10, replace=False)

In [ ]:
# Copy test images
for img in test_rust:
    shutil.copy(f"{base_path}/rust/{img}", f"{test_dir}/rust/{img}")

for img in test_no_rust:
    shutil.copy(f"{base_path}/no_rust/{img}", f"{test_dir}/no_rust/{img}")

In [ ]:
# Copy training images (excluding test images)
for img in rust_images:
    if img not in test_rust:
        shutil.copy(f"{base_path}/rust/{img}", f"{train_dir}/rust/{img}")

for img in no_rust_images:
    if img not in test_no_rust:
        shutil.copy(f"{base_path}/no_rust/{img}", f"{train_dir}/no_rust/{img}")

print("\nData split completed!")
print(f"Training rust images: {len(os.listdir(train_dir + '/rust'))}")
print(f"Training no_rust images: {len(os.listdir(train_dir + '/no_rust'))}")
print(f"Test images: 20 (10 rust + 10 no_rust)")


Data split completed!
Training rust images: 14
Training no_rust images: 14
Test images: 20 (10 rust + 10 no_rust)


CNN Model

In [ ]:
# Prepare data generators
img_height, img_width = 224, 224
batch_size = 8  # Reduced batch size for small dataset

train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

test_datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

Found 24 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


In [ ]:
# Build Simple CNN Model
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_30 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_30 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_31 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_31 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_32 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_32 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_10 (Flatten)            │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Add early stopping and reduce learning rate callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

# Train the model
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr]
)

Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 236ms/step - accuracy: 0.7292 - loss: 0.5801 - val_accuracy: 0.5000 - val_loss: 0.6696 - learning_rate: 5.0000e-05
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 0.5677 - loss: 0.6475 - val_accuracy: 0.5000 - val_loss: 0.6770 - learning_rate: 5.0000e-05
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.7083 - loss: 0.5903 - val_accuracy: 0.5000 - val_loss: 0.6667 - learning_rate: 5.0000e-05
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - accuracy: 0.6354 - loss: 0.5821 - val_accuracy: 0.5000 - val_loss: 0.7010 - learning_rate: 5.0000e-05
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 184ms/step - accuracy: 0.6458 - loss: 0.5836 - val_accuracy: 0.5000 - val_loss: 0.6550 - learning_rate: 5.0000e-05
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.7812 - loss: 0.5685 - val_accuracy: 0.5000 - val_loss: 0.6996 - learning_rate: 5.0000e-05
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - accuracy: 0.6615 - loss:

In [ ]:
# Save the model
model.save('/content/drive/MyDrive/Colab Notebooks/cnn_rust_model.h5')
print("\nModel saved!")


Model saved!


In [ ]:
# Test on the test set
test_results = []
class_names = ['no_rust', 'rust']

for class_name in class_names:
    class_path = os.path.join(test_dir, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        img = keras.preprocessing.image.load_img(img_path, target_size=(img_height, img_width))
        img_array = keras.preprocessing.image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0) / 255.0

        prediction = model.predict(img_array, verbose=0)[0][0]
        predicted_class = 'rust' if prediction > 0.5 else 'no_rust'

        test_results.append({
            'image': img_name,
            'true_class': class_name,
            'predicted_class': predicted_class,
            'confidence': prediction if prediction > 0.5 else 1 - prediction
        })

In [ ]:
# Calculate accuracy
correct = sum(1 for r in test_results if r['true_class'] == r['predicted_class'])
accuracy = correct / len(test_results) * 100

print("\n" + "="*60)
print("CNN MODEL TEST RESULTS")
print("="*60)
print(f"{'Image':<30} {'True Class':<15} {'Predicted Class':<15}")
print("-"*60)
for result in test_results:
    print(f"{result['image']:<30} {result['true_class']:<15} {result['predicted_class']:<15}")
print("-"*60)
print(f"Overall Accuracy: {accuracy:.2f}% ({correct}/20)")
print("="*60)


CNN MODEL TEST RESULTS
Image                          True Class      Predicted Class
------------------------------------------------------------
CA-560759L-6232016-FLD-DC-VIC-005-G03-001_jebeauhz.21x.jpg no_rust         no_rust        
002_yoamnba2.wdq.jpg           no_rust         no_rust        
106_gwxpfsmj.bni.jpg           no_rust         no_rust        
007_drdnswnb.xeb.jpg           no_rust         no_rust        
003_zomak3wq.43c.jpg           no_rust         no_rust        
002_ocpp5ini.zep.jpg           no_rust         no_rust        
002_vzt4zfyf.2j3.jpg           no_rust         no_rust        
002_2bvturws.zss.jpg           no_rust         no_rust        
CA-560759R_14.jpg              no_rust         no_rust        
002_zar1fupp.kf2.jpg           no_rust         no_rust        
IMG_0377_n4suhq5y.zqs.jpg      rust            rust           
002_q40jac40.sd4.jpg           rust            no_rust        
007_1vtpg0jx.nkh.jpg           rust            rust           
RIMG1

In [ ]:
# Save test results to CSV
import pandas as pd
df_results = pd.DataFrame(test_results)
df_results.to_csv('/content/drive/MyDrive/Colab Notebooks/cnn_test_results.csv', index=False)

In [ ]:
# Create output folder and save test images with predictions
output_dir = '/content/drive/MyDrive/Colab Notebooks/cnn_test'
os.makedirs(output_dir, exist_ok=True)

for result in test_results:
    class_name = result['true_class']
    img_name = result['image']
    img_path = os.path.join(test_dir, class_name, img_name)

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Add prediction text
    text = f"Pred: {result['predicted_class']} ({result['confidence']:.2f})"
    color = (0, 255, 0) if result['true_class'] == result['predicted_class'] else (255, 0, 0)
    cv2.putText(img, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(img, f"True: {result['true_class']}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(output_dir, img_name), img)

print(f"\nTest images saved to: {output_dir}")


Test images saved to: /content/drive/MyDrive/Colab Notebooks/cnn_test


ResNet50 Model

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers
import numpy as np
import os
import cv2

In [ ]:
# Use the same train and test directories from previous code
train_dir = '/content/train_data'
test_dir = '/content/test_data'
img_height, img_width = 224, 224
batch_size = 8  # Reduced batch size for small dataset

In [ ]:
# Prepare data generators
train_datagen = keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

Found 24 images belonging to 2 classes.
Found 4 images belonging to 2 classes.


In [ ]:
# Build ResNet50 Model
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(img_height, img_width, 3)
)

# Freeze base model layers
base_model.trainable = False

# Create new model on top
resnet_model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

resnet_model.summary()

Model: "sequential_26"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_15     │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_30 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,112,513 (91.98 MB)

 Trainable params: 524,801 (2.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
# Add early stopping and reduce learning rate callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=0.00001
)

# Train the model
print("\nTraining ResNet50 model...")
history = resnet_model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr]
)



Training ResNet50 model...
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step - accuracy: 0.4115 - loss: 0.8371 - val_accuracy: 0.5000 - val_loss: 0.6937 - learning_rate: 1.0000e-04
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 242ms/step - accuracy: 0.4635 - loss: 0.8448 - val_accuracy: 0.5000 - val_loss: 0.6930 - learning_rate: 1.0000e-04
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 251ms/step - accuracy: 0.5365 - loss: 0.8541 - val_accuracy: 0.5000 - val_loss: 0.6927 - learning_rate: 1.0000e-04
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 240ms/step - accuracy: 0.3906 - loss: 0.8390 - val_accuracy: 0.5000 - val_loss: 0.6910 - learning_rate: 1.0000e-04
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 255ms/step - accuracy: 0.3802 - loss: 0.8306 - val_accuracy: 0.7500 - val_loss: 0.6886 - learning_rate: 1.0000e-04
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 181ms/step - accuracy: 0.4948 - loss: 0.7916 - val_accuracy: 0.7500 - val_loss: 0.6900 - learning_rate: 1.0000e-04
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 180ms/step 

In [ ]:
# Save the model
resnet_model.save('/content/drive/MyDrive/Colab Notebooks/resnet50_rust_model.h5')
print("\nResNet50 model saved!")


ResNet50 model saved!


In [ ]:
# Test on the test set
test_results = []
class_names = ['no_rust', 'rust']

for class_name in class_names:
    class_path = os.path.join(test_dir, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        img = keras.preprocessing.image.load_img(img_path, target_size=(img_height, img_width))
        img_array = keras.preprocessing.image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0) / 255.0

        prediction = resnet_model.predict(img_array, verbose=0)[0][0]
        predicted_class = 'rust' if prediction > 0.5 else 'no_rust'

        test_results.append({
            'image': img_name,
            'true_class': class_name,
            'predicted_class': predicted_class,
            'confidence': prediction if prediction > 0.5 else 1 - prediction
        })

In [ ]:
# Calculate accuracy
correct = sum(1 for r in test_results if r['true_class'] == r['predicted_class'])
accuracy = correct / len(test_results) * 100

print("\n" + "="*60)
print("RESNET50 MODEL TEST RESULTS")
print("="*60)
print(f"{'Image':<30} {'True Class':<15} {'Predicted Class':<15}")
print("-"*60)
for result in test_results:
    print(f"{result['image']:<30} {result['true_class']:<15} {result['predicted_class']:<15}")
print("-"*60)
print(f"Overall Accuracy: {accuracy:.2f}% ({correct}/20)")
print("="*60)


RESNET50 MODEL TEST RESULTS
Image                          True Class      Predicted Class
------------------------------------------------------------
CA-560759L-6232016-FLD-DC-VIC-005-G03-001_jebeauhz.21x.jpg no_rust         rust           
002_yoamnba2.wdq.jpg           no_rust         no_rust        
106_gwxpfsmj.bni.jpg           no_rust         no_rust        
007_drdnswnb.xeb.jpg           no_rust         no_rust        
003_zomak3wq.43c.jpg           no_rust         rust           
002_ocpp5ini.zep.jpg           no_rust         no_rust        
002_vzt4zfyf.2j3.jpg           no_rust         no_rust        
002_2bvturws.zss.jpg           no_rust         no_rust        
CA-560759R_14.jpg              no_rust         no_rust        
002_zar1fupp.kf2.jpg           no_rust         no_rust        
IMG_0377_n4suhq5y.zqs.jpg      rust            no_rust        
002_q40jac40.sd4.jpg           rust            no_rust        
007_1vtpg0jx.nkh.jpg           rust            no_rust        


In [ ]:
# Save test results to CSV
import pandas as pd
df_results = pd.DataFrame(test_results)
df_results.to_csv('/content/drive/MyDrive/Colab Notebooks/resnet50_test_results.csv', index=False)

In [ ]:
# Create output folder and save test images with predictions
output_dir = '/content/drive/MyDrive/Colab Notebooks/resnet50_test'
os.makedirs(output_dir, exist_ok=True)

for result in test_results:
    class_name = result['true_class']
    img_name = result['image']
    img_path = os.path.join(test_dir, class_name, img_name)

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Add prediction text
    text = f"Pred: {result['predicted_class']} ({result['confidence']:.2f})"
    color = (0, 255, 0) if result['true_class'] == result['predicted_class'] else (255, 0, 0)
    cv2.putText(img, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    cv2.putText(img, f"True: {result['true_class']}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(os.path.join(output_dir, img_name), img)

print(f"\nTest images saved to: {output_dir}")


Test images saved to: /content/drive/MyDrive/Colab Notebooks/resnet50_test
